<a href="https://colab.research.google.com/github/sflores14/inspirastem2026-bioquimica-computacional/blob/main/day1/01_de_biologia_a_estructura.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 1 | De una pregunta biológica a una hipótesis molecular

**InspiraSTEM 2026 | Bioquímica Computacional Aplicada**

### Pregunta del día
> **Qué podemos predecir sobre la unión de una molécula usando solamente su estructura y la estructura de una proteína?**

Hoy construiremos una hipótesis paso a paso. Todavía **no buscamos la respuesta experimental**.

**proteína → estructura → cavidad → ligandos → predicción → evidencia → hipótesis**

## 1. Antes de comenzar

Trabajaremos como si estuviéramos resolviendo una pregunta de investigación real:

- haremos predicciones antes de ver resultados;
- utilizaremos herramientas computacionales para obtener nueva evidencia;
- compararemos interpretaciones;
- cambiaremos de opinión si la evidencia lo justifica.

Una predicción computacional es una **hipótesis que podemos evaluar**, no una respuesta experimental.

## 2. Preparar el entorno

La primera ejecución puede tardar unos minutos porque descarga P2Rank.

In [ ]:
#@title Preparar el entorno
import os, sys, glob, tarfile, shutil, subprocess, importlib.util, urllib.request
from pathlib import Path

def ensure_package(package, import_name=None):
    import_name = import_name or package
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )

for pkg, imp in [
    ("rdkit","rdkit"), ("py3Dmol","py3Dmol"), ("pandas","pandas"),
    ("numpy","numpy"), ("requests","requests")
]:
    ensure_package(pkg, imp)

import numpy as np
import pandas as pd
import requests, py3Dmol
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, Crippen, Lipinski
from IPython.display import display

if shutil.which("java") is None:
    subprocess.check_call(["apt-get","update","-qq"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call(["apt-get","install","-y","-qq","openjdk-17-jre-headless"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

archive = Path("/content/p2rank_2.5.1.tar.gz")
extract_dir = Path("/content/p2rank_2.5.1")

if not extract_dir.exists():
    print("Descargando P2Rank 2.5.1...")
    urllib.request.urlretrieve(
        "https://github.com/rdk/p2rank/releases/download/2.5.1/p2rank_2.5.1.tar.gz",
        archive
    )
    extract_dir.mkdir(exist_ok=True)
    with tarfile.open(archive, "r:gz") as tf:
        tf.extractall(extract_dir)

candidates = list(extract_dir.rglob("prank"))
if not candidates:
    raise RuntimeError("No se encontró el ejecutable de P2Rank.")
PRANK = candidates[0]
P2RANK_ROOT = PRANK.parent
os.chmod(PRANK, 0o755)

print("Entorno listo.")

## 3. De la biología a la estructura

Podemos estudiar un fenómeno biológico en muchos niveles:

**organismo → tejido → célula → proteína → molécula → átomos**

Durante el workshop iremos constantemente desde el fenómeno biológico hasta el nivel atómico y de regreso.

## 4. Conozcamos nuestra proteína

Trabajaremos con una variante de **T4 lysozyme** que contiene la mutación:

**L99A = Leu99 → Ala99**

Primero descargaremos una estructura experimental de la proteína sin un ligando ocupando la cavidad que estudiaremos.

In [ ]:
#@title Descargar la estructura
PDB_ID = "4W51"
r = requests.get(f"https://files.rcsb.org/download/{PDB_ID}.pdb", timeout=60)
r.raise_for_status()
raw_pdb = r.text

with open(f"{PDB_ID}.pdb","w") as f:
    f.write(raw_pdb)

# Protein-only chain A
protein_lines = [
    line for line in raw_pdb.splitlines()
    if line.startswith("ATOM") and len(line) > 21 and line[21] == "A"
]
protein_pdb = "\n".join(protein_lines + ["END"]) + "\n"
PROTEIN_FILE = f"{PDB_ID}_protein.pdb"

with open(PROTEIN_FILE,"w") as f:
    f.write(protein_pdb)

print(f"PDB: {PDB_ID}")
print("Cadena utilizada: A")
print(f"Átomos de proteína: {len(protein_lines)}")
print("Se removieron agua y moléculas no proteicas.")

In [ ]:
#@title Explorar la proteína
representation = "Cartoon" #@param ["Cartoon", "Surface", "Cartoon + Surface", "Sticks"]

view = py3Dmol.view(width=800, height=540)
view.addModel(protein_pdb, "pdb")

if representation == "Cartoon":
    view.setStyle({"cartoon":{"color":"spectrum"}})
elif representation == "Surface":
    view.setStyle({"cartoon":{"color":"lightgray"}})
    view.addSurface(py3Dmol.VDW, {"opacity":0.85})
elif representation == "Cartoon + Surface":
    view.setStyle({"cartoon":{"color":"spectrum"}})
    view.addSurface(py3Dmol.VDW, {"opacity":0.30})
else:
    view.setStyle({"stick":{}})

view.zoomTo()
view.show()

### Observa

- `Cartoon` ayuda a reconocer el fold.
- `Surface` ayuda a pensar como una molécula que se aproxima desde el solvente.
- `Sticks` muestra el nivel atómico.

> **Dónde buscarías una región capaz de acomodar una molécula pequeña?**

## 5. Una mutación puede cambiar un espacio molecular

Leucina tiene una cadena lateral más grande que alanina.

> Si reemplazamos una cadena lateral grande por una más pequeña dentro de una proteína, qué podría ocurrir con el espacio local?

In [ ]:
#@title Examinar la posición 99
neighbor_radius = 5.0 #@param {type:"slider", min:3.0, max:8.0, step:0.5}

atoms=[]
for line in protein_pdb.splitlines():
    if line.startswith("ATOM"):
        atoms.append({
            "serial":int(line[6:11]),
            "name":line[12:16].strip(),
            "resname":line[17:20].strip(),
            "chain":line[21].strip(),
            "resi":int(line[22:26]),
            "x":float(line[30:38]), "y":float(line[38:46]), "z":float(line[46:54])
        })
atoms_df=pd.DataFrame(atoms)

a99=atoms_df[(atoms_df.chain=="A") & (atoms_df.resi==99)]
center99=a99[["x","y","z"]].mean().to_numpy()

coords=atoms_df[["x","y","z"]].to_numpy()
dist=np.linalg.norm(coords-center99,axis=1)
near=atoms_df[dist<=neighbor_radius]
near_res=sorted(set(int(x) for x in near.resi))

view=py3Dmol.view(width=800,height=540)
view.addModel(protein_pdb,"pdb")
view.setStyle({"cartoon":{"color":"lightgray","opacity":0.55}})
view.setStyle({"chain":"A","resi":near_res},{"stick":{"radius":0.18}})
view.setStyle({"chain":"A","resi":99},{"stick":{"radius":0.34},"sphere":{"scale":0.35}})
view.addLabel("A99",{"position":{"x":float(center99[0]),"y":float(center99[1]),"z":float(center99[2])},
                     "backgroundOpacity":0.7})
view.zoomTo({"chain":"A","resi":near_res})
view.show()

## 6. Conozcamos nuestros tres ligandos

Estudiaremos:

- **benzene**
- **toluene**
- **n-propylbenzene**

Todavía no veremos ninguna estructura experimental proteína–ligando.

In [ ]:
#@title Visualizar los ligandos
ligands={
    "Benzene":"c1ccccc1",
    "Toluene":"Cc1ccccc1",
    "n-Propylbenzene":"CCCc1ccccc1"
}
mols=[Chem.MolFromSmiles(s) for s in ligands.values()]
display(Draw.MolsToGridImage(mols,molsPerRow=3,subImgSize=(300,240),legends=list(ligands)))

### Explora los ligandos en 3D

Selecciona una molécula y usa el mouse para:

- rotarla;
- acercar y alejar;
- comparar su forma;
- observar qué tan rígida o flexible parece.

Todavía no estamos colocando el ligando dentro de la proteína. Solo estamos explorando su geometría molecular.

In [ ]:
#@title Explorar un ligando en 3D

ligand_3d = "n-Propylbenzene" #@param ["Benzene", "Toluene", "n-Propylbenzene"]

from rdkit.Chem import AllChem

mol3d = Chem.AddHs(Chem.MolFromSmiles(ligands[ligand_3d]))

params = AllChem.ETKDGv3()
params.randomSeed = 2026
AllChem.EmbedMolecule(mol3d, params)
AllChem.MMFFOptimizeMolecule(mol3d)

molblock = Chem.MolToMolBlock(mol3d)

view = py3Dmol.view(width=620, height=420)
view.addModel(molblock, "sdf")
view.setStyle({
    "stick": {"radius": 0.18},
    "sphere": {"scale": 0.28}
})
view.zoomTo()
view.show()

rotatable = Lipinski.NumRotatableBonds(Chem.RemoveHs(mol3d))
print(ligand_3d)
print(f"Enlaces rotables: {rotatable}")

### Compara

Cambia entre los tres ligandos y fíjate en:

- cuál es más compacto;
- cuál es más alargado;
- cuál tiene más libertad para cambiar de conformación.

> Si una cavidad es pequeña y rígida, qué ventaja o desventaja podría tener un ligando más flexible?

> **Qué cambia de izquierda a derecha?**

Piensa en tamaño, hidrofobicidad, flexibilidad y forma.

In [ ]:
#@title Comparar propiedades moleculares
rows=[]
for name,smi in ligands.items():
    mol=Chem.MolFromSmiles(smi)
    rows.append({
        "Ligando":name,
        "MW (Da)":round(Descriptors.MolWt(mol),2),
        "Átomos pesados":mol.GetNumHeavyAtoms(),
        "MolLogP":round(Crippen.MolLogP(mol),2),
        "Enlaces rotables":Lipinski.NumRotatableBonds(mol)
    })
ligand_df=pd.DataFrame(rows)
display(ligand_df)

### Discute

- Qué propiedad cambia más claramente?
- Más grande significa necesariamente mejor unión?
- Qué podría ocurrir si una molécula se vuelve demasiado grande para una cavidad?

## 7. Tu primera hipótesis

Haz tu predicción **antes** de utilizar un algoritmo de cavidades.

In [ ]:
#@title Tu primera hipótesis
primer_lugar = "n-Propylbenzene" #@param ["Benzene", "Toluene", "n-Propylbenzene"]
segundo_lugar = "Toluene" #@param ["Benzene", "Toluene", "n-Propylbenzene"]
tercer_lugar = "Benzene" #@param ["Benzene", "Toluene", "n-Propylbenzene"]
razon_principal = "Complementariedad de forma" #@param ["Tamaño", "Hidrofobicidad", "Flexibilidad", "Complementariedad de forma", "Otra"]

ranking=[primer_lugar,segundo_lugar,tercer_lugar]
if len(set(ranking))<3:
    print("Revisa tu ranking: usa cada ligando una sola vez.")
else:
    print("HIPÓTESIS INICIAL")
    for i,x in enumerate(ranking,1):
        print(f"{i}. {x}")
    print("Razón principal:",razon_principal)

## 8. Dónde podría ocurrir la unión?

**Cavidad:** región geométrica de la superficie o interior de una proteína.

**Binding site:** región donde una molécula realmente interactúa con la proteína.

**Predicted pocket:** región que un método computacional considera compatible con un sitio de unión.

> **Son necesariamente la misma cosa?**

## 9. P2Rank: predicción de cavidades

P2Rank analiza la superficie accesible al solvente y utiliza machine learning para identificar regiones con características similares a sitios de unión conocidos.

Nuestra pregunta es:

> **Qué regiones de esta estructura parecen cavidades plausibles para unión de ligandos?**

In [ ]:
#@title Ejecutar P2Rank
OUTPUT_DIR=Path("/content/p2rank_output")
OUTPUT_DIR.mkdir(exist_ok=True)

cmd=[
    str(PRANK),"predict",
    "-f",str(Path(PROTEIN_FILE).resolve()),
    "-o",str(OUTPUT_DIR.resolve()),
    "-visualizations","0"
]

result=subprocess.run(
    cmd,cwd=str(P2RANK_ROOT),text=True,
    stdout=subprocess.PIPE,stderr=subprocess.STDOUT
)

if result.returncode!=0:
    print(result.stdout[-4000:])
    raise RuntimeError("P2Rank no terminó correctamente.")

prediction_files=list(OUTPUT_DIR.rglob("*_predictions.csv"))
if not prediction_files:
    raise FileNotFoundError("No se encontró el archivo de predicciones.")

P2RANK_PREDICTIONS=prediction_files[0]
pockets=pd.read_csv(P2RANK_PREDICTIONS,skipinitialspace=True)
pockets.columns=[c.strip() for c in pockets.columns]

cols=[c for c in ["rank","score","probability","residue_ids"] if c in pockets.columns]
display(pockets[cols].head(5))
print(f"P2Rank encontró {len(pockets)} cavidades candidatas.")

### Interpreta, no solo leas el ranking

- `rank` = posición según el modelo;
- `score` = puntuación del modelo;
- `probability` = probabilidad calibrada reportada;
- `residue_ids` = residuos asociados.

> **El pocket con mayor score no tiene por qué ser automáticamente el sitio biológicamente relevante.**

In [ ]:
#@title Explorar las cavidades predichas
pocket_rank = 1 #@param {type:"slider", min:1, max:5, step:1}
show_surface = True #@param {type:"boolean"}

row=pockets.iloc[pocket_rank-1]

def parse_p2rank_residues(value):
    result=[]
    if pd.isna(value):
        return result
    for token in str(value).split():
        if "_" in token:
            chain,num=token.split("_",1)
            try:
                result.append((chain,int(num)))
            except:
                pass
    return result

pocket_res=parse_p2rank_residues(row.get("residue_ids",""))
res_by_chain={}
for chain,resi in pocket_res:
    res_by_chain.setdefault(chain,[]).append(resi)

cx,cy,cz=[float(row[k]) for k in ["center_x","center_y","center_z"]]

view=py3Dmol.view(width=800,height=540)
view.addModel(protein_pdb,"pdb")
view.setStyle({"cartoon":{"color":"lightgray","opacity":0.65}})
if show_surface:
    view.addSurface(py3Dmol.VDW,{"opacity":0.25})

for chain,residues in res_by_chain.items():
    view.setStyle({"chain":chain,"resi":residues},{"stick":{"radius":0.22}})

view.addSphere({"center":{"x":cx,"y":cy,"z":cz},"radius":1.0,"color":"yellow","opacity":0.85})
view.addLabel(f"Pocket {pocket_rank}",{"position":{"x":cx,"y":cy,"z":cz},"backgroundOpacity":0.75})

target=[r for c,r in pocket_res if c=="A"]
if target:
    view.zoomTo({"chain":"A","resi":target})
else:
    view.zoomTo()
view.show()

print(f"Pocket {pocket_rank}")
if "score" in row: print("Score:",round(float(row["score"]),3))
if "probability" in row: print("Probability:",round(float(row["probability"]),3))

### Explora

Cambia `pocket_rank` entre 1 y 5.

- parece suficientemente grande?
- está enterrada o expuesta?
- está cerca de A99?
- parece compatible con los tres ligandos?

**Mira la estructura, no solamente la tabla.**

## 10. Tu grupo decide

Selecciona la cavidad que tu grupo considera más plausible.

In [ ]:
#@title Seleccionar una cavidad candidata
selected_pocket = 1 #@param {type:"slider", min:1, max:5, step:1}

row=pockets.iloc[selected_pocket-1]
print(f"Cavidad seleccionada: Pocket {selected_pocket}")
if "score" in row: print("P2Rank score:",round(float(row["score"]),3))
if "probability" in row: print("P2Rank probability:",round(float(row["probability"]),3))

## 11. Qué residuos forman el entorno?

Incluso una frase aparentemente sencilla como “residuos del binding site” requiere una definición operacional.

Vamos a variar la distancia utilizada para definir el entorno molecular.

In [ ]:
#@title Definir el entorno molecular
distance_cutoff = 4.5 #@param {type:"slider", min:3.0, max:7.0, step:0.5}

row=pockets.iloc[selected_pocket-1]
center=np.array([float(row["center_x"]),float(row["center_y"]),float(row["center_z"])])

coords=atoms_df[["x","y","z"]].to_numpy()
distances=np.linalg.norm(coords-center,axis=1)
local_atoms=atoms_df[distances<=distance_cutoff]

local_residues=(
    local_atoms[["chain","resi","resname"]]
    .drop_duplicates()
    .sort_values(["chain","resi"])
)

res_list=[int(x) for x in local_residues[local_residues.chain=="A"].resi]

view=py3Dmol.view(width=800,height=540)
view.addModel(protein_pdb,"pdb")
view.setStyle({"cartoon":{"color":"lightgray","opacity":0.55}})
view.setStyle({"chain":"A","resi":res_list},{"stick":{"radius":0.24}})
view.addSphere({"center":{"x":float(center[0]),"y":float(center[1]),"z":float(center[2])},
                "radius":0.9,"color":"yellow"})
view.zoomTo({"chain":"A","resi":res_list})
view.show()

print(f"Cutoff: {distance_cutoff:.1f} Å")
print(f"Residuos incluidos: {len(local_residues)}")
display(local_residues.reset_index(drop=True))

### Experimento del grupo

- **Grupo 1:** 3.5 Å
- **Grupo 2:** 4.0 Å
- **Grupo 3:** 4.5 Å
- **Grupo 4:** 5.5 Å
- **Grupo 5:** 6.5 Å

Comparen:

- cuántos residuos aparecen?
- qué tipos de aminoácidos se incorporan?
- en qué momento el cutoff empieza a incluir residuos que probablemente no interactúan directamente?

> **Las decisiones de análisis también forman parte del método científico.**

## 12. Pintar la química del pocket

Ahora vamos a convertir la lista de residuos en una imagen química del pocket.

La clasificación es simple y solo sirve para interpretar el ambiente local.

In [ ]:
#@title Visualizar la química del pocket

chemistry_cutoff = 5.0 #@param {type:"slider", min:3.0, max:7.0, step:0.5}

classes = {
    "Hidrofóbico": {"ALA","VAL","LEU","ILE","MET","PRO"},
    "Aromático": {"PHE","TYR","TRP"},
    "Polar": {"SER","THR","ASN","GLN","CYS"},
    "Positivo": {"LYS","ARG","HIS"},
    "Negativo": {"ASP","GLU"},
    "Especial": {"GLY"}
}

class_colors = {
    "Hidrofóbico": "orange",
    "Aromático": "purple",
    "Polar": "cyan",
    "Positivo": "blue",
    "Negativo": "red",
    "Especial": "green",
    "Otro": "gray"
}

def residue_class(resname):
    for label, members in classes.items():
        if resname in members:
            return label
    return "Otro"

row = pockets.iloc[selected_pocket - 1]
center = np.array([
    float(row["center_x"]),
    float(row["center_y"]),
    float(row["center_z"])
])

coords = atoms_df[["x","y","z"]].to_numpy()
distances = np.linalg.norm(coords - center, axis=1)
local_atoms = atoms_df[distances <= chemistry_cutoff]

chem_env = (
    local_atoms[["chain","resi","resname"]]
    .drop_duplicates()
    .sort_values(["chain","resi"])
)
chem_env["Clase"] = chem_env["resname"].map(residue_class)

view = py3Dmol.view(width=800, height=540)
view.addModel(protein_pdb, "pdb")
view.setStyle({"cartoon":{"color":"lightgray","opacity":0.40}})

for label, color in class_colors.items():
    residues = [
        int(x) for x in chem_env.loc[
            (chem_env["Clase"] == label) & (chem_env["chain"] == "A"), "resi"
        ]
    ]
    if residues:
        view.setStyle(
            {"chain":"A","resi":residues},
            {"stick":{"color":color,"radius":0.24}}
        )

view.addSphere({
    "center":{"x":float(center[0]),"y":float(center[1]),"z":float(center[2])},
    "radius":0.9,
    "color":"yellow",
    "opacity":0.75
})
view.zoomTo({"chain":"A","resi":[int(x) for x in chem_env.loc[chem_env.chain=="A","resi"]]})
view.show()

summary = (
    chem_env["Clase"]
    .value_counts()
    .rename_axis("Clase")
    .reset_index(name="Número de residuos")
)
display(summary)

### Discute con tu grupo

- Qué tipo de residuos dominan el entorno?
- Ese ambiente parece compatible con nuestros tres ligandos?
- Qué información todavía nos falta?

## 13. Comparar los ligandos dentro del mismo espacio

Vamos a colocar cada ligando en el centro del pocket solamente para comparar **tamaño y forma**.

Esto **no es docking** y no representa una pose predicha.

In [ ]:
#@title Comparar un ligando con el pocket

ligand_to_view = "n-Propylbenzene" #@param ["Benzene", "Toluene", "n-Propylbenzene"]

from rdkit.Chem import AllChem

mol = Chem.AddHs(Chem.MolFromSmiles(ligands[ligand_to_view]))
params = AllChem.ETKDGv3()
params.randomSeed = 2026
AllChem.EmbedMolecule(mol, params)
AllChem.MMFFOptimizeMolecule(mol)

conf = mol.GetConformer()
xyz = np.array([
    [conf.GetAtomPosition(i).x, conf.GetAtomPosition(i).y, conf.GetAtomPosition(i).z]
    for i in range(mol.GetNumAtoms())
])

shift = center - xyz.mean(axis=0)

for i in range(mol.GetNumAtoms()):
    p = conf.GetAtomPosition(i)
    conf.SetAtomPosition(i, (p.x + shift[0], p.y + shift[1], p.z + shift[2]))

view = py3Dmol.view(width=800, height=540)
view.addModel(protein_pdb, "pdb")
view.setStyle({"model":0},{"cartoon":{"color":"lightgray","opacity":0.50}})
view.addSurface(py3Dmol.VDW, {"opacity":0.5}, {"model":0})

view.addModel(Chem.MolToMolBlock(mol), "sdf")
view.setStyle({"model":1},{"stick":{"radius":0.22},"sphere":{"scale":0.25}})

view.zoomTo({"model":1})
view.show()

print(ligand_to_view)
print("Visualización geométrica solamente — no es docking.")

Cambia entre los tres ligandos.

> Si solo miráramos tamaño y forma, cuál parece más fácil de acomodar?

Luego piensa en la limitación:

> Que una molécula quepa significa que se va a unir mejor?

Mañana podremos empezar a responder eso con predicciones de poses e interacciones.

## 14. Hipótesis final del Día 1

Usa lo que vimos hoy:

- propiedades de los ligandos;
- mutación L99A;
- pocket predicho;
- residuos alrededor del pocket;
- ambiente químico;
- tamaño relativo de los ligandos.

No necesitas estar seguro. Lo importante es poder explicar por qué elegiste tu ranking.

In [ ]:
#@title Hipótesis del grupo

final_1 = "n-Propylbenzene" #@param ["Benzene", "Toluene", "n-Propylbenzene"]
final_2 = "Toluene" #@param ["Benzene", "Toluene", "n-Propylbenzene"]
final_3 = "Benzene" #@param ["Benzene", "Toluene", "n-Propylbenzene"]

evidencia_principal = "Combinación de evidencia" #@param ["Tamaño molecular", "Hidrofobicidad", "Flexibilidad", "Predicción de P2Rank", "Ambiente químico del pocket", "Combinación de evidencia"]

confianza = 3 #@param {type:"slider", min:1, max:5, step:1}

ranking = [final_1, final_2, final_3]

if len(set(ranking)) < 3:
    print("Revisa tu ranking: usa cada ligando una sola vez.")
else:
    from IPython.display import HTML

    display(HTML(f'''
    <div style="border:1px solid #bbb; border-radius:12px; padding:18px; max-width:620px;">
      <h3 style="margin-top:0;">Día 1 — Hipótesis del grupo</h3>
      <p><b>Pocket:</b> {selected_pocket}</p>
      <p><b>Ranking:</b><br>
      1. {final_1}<br>
      2. {final_2}<br>
      3. {final_3}</p>
      <p><b>Evidencia principal:</b> {evidencia_principal}</p>
      <p><b>Confianza:</b> {confianza}/5</p>
    </div>
    '''))

## Cierre

Hoy llegamos hasta:

**estructura → pocket → química → hipótesis**

Pero todavía no sabemos **cómo se orienta cada ligando dentro del pocket**.

Ese será el problema del Día 2.

---

## Herramientas y referencias

- RCSB Protein Data Bank
- P2Rank 2.5.1
- RDKit
- py3Dmol

**P2Rank:** Krivák R, Hoksza D. *J Cheminform.* 2018;10:39. DOI: 10.1186/s13321-018-0285-8

La organización del workflow y varias decisiones de diseño para Google Colab están inspiradas en enfoques reproducibles de **Cloud-Bind** y otros recursos abiertos de quimioinformática estructural.